# Spiking Transformer U-Net with LLM Agentic Triage (Phi-3-Mini)

This notebook implements the training pipeline for the BraTS 2023 Glioma dataset using a custom Spiking Transformer U-Net architecture.
Crucially, it replaces a standard mathematical entropy threshold with an **LLM Orchestrator (Phi-3-Mini)**. The LLM acts as an Agentic Triage system, deciding whether a 3D MRI patch is complex enough to require high-precision computing ($T=4$) or if a low-power sweep ($T=1$) suffices.

### Phases Covered:
1. **3D NIfTI Data Ingestion & Target Engineering**
2. **Spiking Tokenizer**
3. **Agentic Triage via Phi-3-Mini** (Dynamic Routing)
4. **Spiking Transformer Encoder** (Ternary Spikes, Shiftmax, BSA)
5. **Spiking U-Net Decoder**
6. **FPTT Optimization Loop**


In [12]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install -q snntorch nibabel transformers accelerate bitsandbytes

In [14]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [15]:
from google.colab import userdata
userdata.get('HF_TOKEN')

'hf_REDACTED'

In [16]:
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

## Phase 1, 2, 4, 5: Core Spiking Architecture Blocks
(Data Loader, Tokenizer, Transformer, Surrogate Gradients, Decoder)

In [17]:
# Constants
DATA_DIR = "/content/drive/MyDrive/datasets/brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
PATCH_SIZE = (64, 64, 64)

class BraTS3DDataset(Dataset):
    def __init__(self, patient_dirs, patch_size=PATCH_SIZE):
        self.patient_dirs = patient_dirs
        self.patch_size = patch_size

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]

        t1n_path = glob.glob(os.path.join(patient_dir, "*-t1n.nii*"))[0]
        t1c_path = glob.glob(os.path.join(patient_dir, "*-t1c.nii*"))[0]
        t2w_path = glob.glob(os.path.join(patient_dir, "*-t2w.nii*"))[0]
        t2f_path = glob.glob(os.path.join(patient_dir, "*-t2f.nii*"))[0]
        seg_path = glob.glob(os.path.join(patient_dir, "*-seg.nii*"))[0]

        t1n = nib.load(t1n_path).get_fdata().astype(np.float32)
        t1c = nib.load(t1c_path).get_fdata().astype(np.float32)
        t2w = nib.load(t2w_path).get_fdata().astype(np.float32)
        t2f = nib.load(t2f_path).get_fdata().astype(np.float32)
        seg = nib.load(seg_path).get_fdata().astype(np.float32)

        image_4d = np.transpose(np.stack([t1n, t1c, t2w, t2f], axis=0), (0, 3, 1, 2))
        seg_3d = np.transpose(seg, (2, 0, 1))

        d, h, w = image_4d.shape[1:]
        pd, ph, pw = self.patch_size

        if d < pd or h < ph or w < pw:
            image_4d = np.pad(image_4d, ((0,0), (0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            seg_3d = np.pad(seg_3d, ((0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            d, h, w = image_4d.shape[1:]

        z = np.random.randint(0, d - pd + 1)
        y = np.random.randint(0, h - ph + 1)
        x = np.random.randint(0, w - pw + 1)

        image_patch = image_4d[:, z:z+pd, y:y+ph, x:x+pw]
        seg_patch = seg_3d[z:z+pd, y:y+ph, x:x+pw]

        wt_mask = (seg_patch > 0).astype(np.float32)
        tc_mask = np.logical_or(seg_patch == 1, seg_patch == 3).astype(np.float32)
        et_mask = (seg_patch == 3).astype(np.float32)
        mask_channels = np.stack([wt_mask, tc_mask, et_mask], axis=0)

        for c in range(4):
            img_c = image_patch[c]
            if img_c.max() > 0:
                image_patch[c] = (img_c - img_c[img_c > 0].mean()) / (img_c[img_c > 0].std() + 1e-8)

        return {'image': torch.from_numpy(image_patch), 'mask': torch.from_numpy(mask_channels)}

class TernarySurrogate(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, threshold=1.0):
        ctx.save_for_backward(input)
        ctx.threshold = threshold
        out = torch.zeros_like(input)
        out[input >= threshold] = 1.0
        out[input <= -threshold] = -1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        return grad_output * torch.exp(-torch.abs(input)), None

class TernaryLIF(nn.Module):
    def __init__(self, beta=0.9, threshold=1.0):
        super().__init__()
        self.beta = beta
        self.threshold = threshold
        self.register_buffer("mem", None)

    def reset_state(self):
        self.mem = None

    def detach_state(self):
        if self.mem is not None:
            self.mem.detach_()

    def forward(self, x):
        if self.mem is None or self.mem.shape != x.shape:
            self.mem = torch.zeros_like(x)
        self.mem = self.beta * self.mem + x
        spk = TernarySurrogate.apply(self.mem, self.threshold)
        self.mem = self.mem - spk * self.threshold
        return spk


In [18]:
def shiftmax(x, dim=-1):
    x_norm = x - torch.max(x, dim=dim, keepdim=True)[0]
    approx_exp = 2.0 ** torch.floor(x_norm)
    return approx_exp / (torch.sum(approx_exp, dim=dim, keepdim=True) + 1e-8)

class BipolarSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.head_dim = embed_dim // num_heads
        self.num_heads = num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.lif_q = TernaryLIF()
        self.lif_k = TernaryLIF()
        self.lif_v = TernaryLIF()

    def reset_states(self):
        self.lif_q.reset_state()
        self.lif_k.reset_state()
        self.lif_v.reset_state()

    def forward(self, x):
        B, S, E = x.shape
        q_spk = self.lif_q(self.q_proj(x))
        k_spk = self.lif_k(self.k_proj(x))
        v_spk = self.lif_v(self.v_proj(x))

        q_head = q_spk.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k_head = k_spk.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v_head = v_spk.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        attn_weights = shiftmax(torch.matmul(q_head, k_head.transpose(-2, -1)) / (self.head_dim ** 0.5), dim=-1)
        attn_out = torch.matmul(attn_weights, v_head).transpose(1, 2).contiguous().view(B, S, E)
        return self.out_proj(attn_out)

class SpikingTokenizer3D(nn.Module):
    def __init__(self, in_channels=4, embed_dim=32):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, embed_dim // 2, kernel_size=3, stride=2, padding=1)
        self.lif1 = TernaryLIF()
        self.conv2 = nn.Conv3d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1)
        self.lif2 = TernaryLIF()

    def reset_states(self):
        self.lif1.reset_state()
        self.lif2.reset_state()
    def forward(self, x):
        s1 = self.lif1(self.conv1(x))
        return self.lif2(self.conv2(s1)), s1

class SpikingTransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, depth):
        super().__init__()
        self.layers = nn.ModuleList([BipolarSelfAttention(embed_dim, num_heads) for _ in range(depth)])
    def reset_states(self):
        for layer in self.layers: layer.reset_states()
    def forward(self, x):
        B, E, D, H, W = x.shape
        x_flat = x.view(B, E, D*H*W).transpose(1, 2)
        for layer in self.layers: x_flat = x_flat + layer(x_flat)
        return x_flat.transpose(1, 2).view(B, E, D, H, W)

class SpikingUNetDecoder3D(nn.Module):
    def __init__(self, embed_dim, out_channels=3):
        super().__init__()
        self.upconv1 = nn.ConvTranspose3d(embed_dim, embed_dim // 2, kernel_size=4, stride=2, padding=1)
        self.lif_up1 = TernaryLIF()
        self.upconv2 = nn.ConvTranspose3d(embed_dim // 2, out_channels, kernel_size=4, stride=2, padding=1)
    def reset_states(self):
        self.lif_up1.reset_state()
    def forward(self, encoder_spk, skip_spk):
        return self.upconv2(self.lif_up1(self.upconv1(encoder_spk) + skip_spk))


## Phase 3: Phi-3-Mini Orchestrator (LLM Agentic Triage)
Loading Phi-3 in 4-bit to fit into Colab's T4 alongside the 3D Sub-network.

In [19]:
class Phi3TriageAgent:
    """
    Loads Phi-3-Mini-Instruct via transformers and bitsandbytes to operate entirely
    within Colab's GPU constraint (taking ~2.5GB VRAM in 4-bit mode).
    """
    def __init__(self):
        print("Initializing Phi-3-Mini Triage Agent (4-bit loading)...")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
        self.model = AutoModelForCausalLM.from_pretrained(
            "microsoft/Phi-3-mini-4k-instruct",
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=False
        )
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
        )
        print("Agent Initialized.")

    def decide_routing(self, entropy_score, foreground_pixels):
        """
        Prompts Phi-3 with the batch's metrics. Returns True for high-precision, False for low.
        """
        prompt = f"""<|user|>
You are a medical imaging orchestrator. You must decide whether to route a 3D MRI patch to normal mode (T=1) or high-precision mode (T=4).
Current Patch Metrics:
- Model Uncertainty (Predictive Entropy): {entropy_score:.4f} (Threshold to worry is ~0.50)
- Estimated Tumor Boundaries (Foreground Pixels): {foreground_pixels:.2f}

If the uncertainty is high OR there is a significant amount of complex foreground boundaries, reply exactly with 'ROUTE_HIGH'.
If the patch seems simple or background-dominated, reply exactly with 'ROUTE_LOW'.
Reply strictly with only ONE of the two options without any explanations.<|end|>
<|assistant|>"""

        # Generate response
        outputs = self.pipe(prompt, max_new_tokens=5, temperature=0.1, do_sample=False)
        text = outputs[0]['generated_text']
        response = text.split("<|assistant|>")[-1].strip().upper()

        if "ROUTE_HIGH" in response:
            return True
        return False


In [20]:
class AgenticSpikingTransformerLLM(nn.Module):
    def __init__(self, agent_llm, in_channels=4, out_channels=3, embed_dim=32, num_heads=4, depth=2):
        super().__init__()
        self.tokenizer = SpikingTokenizer3D(in_channels, embed_dim)
        self.encoder = SpikingTransformerEncoder(embed_dim, num_heads, depth)
        self.decoder = SpikingUNetDecoder3D(embed_dim, out_channels)
        self.agent_llm = agent_llm

    def reset_all_states(self):
        self.tokenizer.reset_states()
        self.encoder.reset_states()
        self.decoder.reset_states()

    def forward_step(self, x):
        enc_spk, skip_spk = self.tokenizer(x)
        return self.decoder(self.encoder(enc_spk), skip_spk)

    def forward(self, x, force_T=None):
        if force_T is not None:
            out_potentials = []
            self.reset_all_states()
            for t in range(force_T):
                out_potentials.append(self.forward_step(x))
            return torch.stack(out_potentials, dim=0)

        self.reset_all_states()

        # Low-Power Screening (T=1)
        mem_t1 = self.forward_step(x)
        probs_t1 = torch.sigmoid(mem_t1)

        # Get metrics for Phi-3
        entropy = -probs_t1 * torch.log(probs_t1 + 1e-6) - (1-probs_t1) * torch.log(1-probs_t1 + 1e-6)
        mean_entropy = entropy.mean().item()
        fg_pixels = (probs_t1 > 0.5).sum().item()

        # Agentic Evaluation via LLM
        route_high = self.agent_llm.decide_routing(mean_entropy, fg_pixels)

        if route_high:
            print(f"> LLM routed HIGH (Entropy: {mean_entropy:.3f}, FG: {fg_pixels})")
            mem_sum = mem_t1
            for _ in range(3):
                mem_sum += self.forward_step(x)
            return (mem_sum / 4.0).unsqueeze(0)
        else:
            # Fast Exit (T=1)
            print(f"> LLM routed LOW (Entropy: {mean_entropy:.3f})")
            return mem_t1.unsqueeze(0)


## Phase 6: Memory-Efficient FPTT Loop


In [21]:
def detach_states(model):
    for module in model.modules():
        if isinstance(module, TernaryLIF):
            module.detach_state()

def train_fptt_llm(model, dataloader, optimizer, criterion, epochs=5, T_train=4, trunc_steps=2):
    model.to(device)
    scaler = torch.cuda.amp.GradScaler()
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            model.reset_all_states()
            batch_loss = 0
            loss_accum = 0
            optimizer.zero_grad()
            
            for t in range(T_train):
                with torch.cuda.amp.autocast():
                    out_mem = model.forward_step(images)
                    loss = criterion(out_mem, masks)
                loss_accum += loss
                batch_loss += loss.item()
                
                if (t + 1) % trunc_steps == 0 or (t + 1) == T_train:
                    scaler.scale(loss_accum).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    loss_accum = 0
                    detach_states(model)
                    del out_mem, loss
                    torch.cuda.empty_cache()
                    
            epoch_loss += batch_loss / T_train
            print(f"Epoch [{epoch+1}/{epochs}] | Batch {batch_idx+1}/{len(dataloader)} | Avg Loss: {batch_loss/T_train:.4f}")
            
        print(f"==> Epoch {epoch+1} Completed. Loss: {epoch_loss/len(dataloader):.4f}")


In [22]:
# Execution Block
patient_dirs = [os.path.join(DATA_DIR, d) for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

if len(patient_dirs) > 0:
    # Initialize Phi-3 orchestrator once
    llm_agent = Phi3TriageAgent()

    train_dataset = BraTS3DDataset(patient_dirs=patient_dirs[:4])
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

    model = AgenticSpikingTransformerLLM(llm_agent, in_channels=4, out_channels=3, embed_dim=32, num_heads=4, depth=2)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    print("Starting LLM-Guided FPTT Memory-efficient Training Loop...")
    train_fptt_llm(model, train_loader, optimizer, criterion, epochs=2, T_train=4, trunc_steps=2)

    print("Testing LLM Forward Pass Inference Integration....")
    sample = train_dataset[0]
    img = sample['image'].unsqueeze(0).to(device)
    # This explicit call triggers the Phi-3 prompt logic on inference
    model.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            out = model(img)
        torch.cuda.empty_cache()
else:
    print("Dataset path not valid or empty. Please mount the google drive securely and retry.")
print("。.:☆*:・'(*⌒―⌒*))) Training Pipeline with Phi-3 Orchestrator successfully configured!")


Initializing Phi-3-Mini Triage Agent (4-bit loading)...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Agent Initialized.
Starting LLM-Guided FPTT Memory-efficient Training Loop...


/tmp/ipykernel_1383/1011438653.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1383/1011438653.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


UnboundLocalError: cannot access local variable 'out_mem' where it is not associated with a value